##**1. Install Library**

In [1]:
!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.0/285.0 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 117.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00


##**2. Import Library**

In [2]:
from roboflow import Roboflow
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


##**3. Download Dataset dari Roboflow**

In [3]:
rf = Roboflow(api_key="dawUk76Tu94kumc2bYJp")
project = rf.workspace("sevenn").project("project-object-detection-telur")
version = project.version(2)
dataset = version.download("yolov11")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to project-object-detection-telur-2 in yolov11:: 100%|██████████| 3838/3838 [00:00<00:00, 7498.97it/s]


In [4]:
with open(f"{dataset.location}/data.yaml", "r") as file:
    print(file.read())

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 2
names: ['cleanshell', 'dirty_shell']

roboflow:
  workspace: sevenn
  project: project-object-detection-telur
  version: 2
  license: Private
  url: https://app.roboflow.com/sevenn/project-object-detection-telur/2


##**4. Load Model YOLO11**

In [5]:
model = YOLO("yolo11n.pt")

##**5. Melatih Model**

In [6]:
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=100,
    patience=10,
    imgsz=640,
    batch=16,
    optimizer="auto",
    device=0,
    workers=2,
    amp=True,
    verbose=True
)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/project-object-detection-telur-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, n

##**6. Memuat Model Terbaik**

In [7]:
best_model = YOLO("/content/runs/detect/train/weights/best.pt")

##**7. Evaluasi Model Menggunakan Data Test**

In [8]:
metrics = best_model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"
)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 616.1±197.0 MB/s, size: 25.7 KB)
val: Scanning /content/project-object-detection-telur-2/test/labels... 69 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 69/69 844.2it/s 0.1s
val: New cache created: /content/project-object-detection-telur-2/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.2s
                   all         69        103      0.998      0.979      0.985      0.979
            cleanshell         35         53      0.997          1      0.995      0.995
           dirty_shell         34         50          1      0.958      0.974      0.962
Speed: 11.4ms preprocess, 18.6ms inference, 0.0ms loss, 3.1ms postprocess per image
Results saved to /content/runs/detect/val


##**8. Menampilkan Metrik Evaluasi**

In [9]:
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")

Precision : 0.9983
Recall    : 0.9789
mAP50     : 0.9846
mAP50-95  : 0.9786


##**9. Inference**

In [10]:
results = best_model.predict(
    source=f"{dataset.location}/test/images",
    conf=0.25,
    save=True,
    verbose=False
)

print(f"Jumlah gambar yang diproses: {len(results)}")
print("Hasil inferensi telah disimpan di:")
print("/content/runs/detect/predict")

Results saved to /content/runs/detect/predict
Jumlah gambar yang diproses: 69
Hasil inferensi telah disimpan di:
/content/runs/detect/predict


##**10. Menyimpan Model**

In [11]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [12]:
!cp -r "/content/runs/detect/train" "/content/drive/MyDrive/Model Object Detection/train"

In [13]:
!cp -r "/content/runs/detect/predict" "/content/drive/MyDrive/Model Object Detection/predict"